# Sistema di sincronizzazione file
Sincronizzazione efficiente tra due cartelle con multiprocessing e multithreading

---

## 1. Introduzione e architettura

L'obiettivo è costruire una architettura in grado di garantire che i file critici siano sempre aggiornati tra una cartella sorgente e una di destinazione, evitando perdita di file aggiornati, duplicazione di dati e lentezza su grandi volumi

### Architettura ibrida multiprocessing + multithreading

```                    
                                                     
  FASE 1: Scansione (multithreading)                 
  ------------  ------------  ------------          
  │ Thread 1 │  │ Thread 2 │  │ Thread N │  ...     
  │ legge    │  │ legge    │  │ legge    │          
  │ metadata │  │ metadata │  │ metadata │          
  ------------  ------------  ------------          
       |--------------|---------------|              
                      V                               
  FASE 2: Categorizzazione                           
  [nuovi] [modificati] [obsoleti]                    
                     │                               
                     V                               
  FASE 3: Copia/eliminazione (multiprocessing)       
  ------------  ------------  ------------          
  │ Process1 │  │ Process2 │  │ Process3 │  ...      
  │ chunk A  │  │ chunk B  │  │ chunk C  │          
  ------------  ------------  ------------          

```

### Scelte architetturali
| Componente | Tecnologia | Motivazione |
|---|---|---|
| Scansione metadati | ThreadPoolExecutor | I/O-bound: il GIL non è un vincolo per operazioni di lettura disco |
| Copia file | ProcessPoolExecutor | CPU/I/O intensive: il multiprocessing bypassa il GIL per carichi pesanti |
| Eliminazione | Thread singolo | Operazioni rapide, overhead di processo non giustificato |
| Gestione sottocartelle | Ricorsione + os.walk | Navigazione completa dell'albero di directory |

---
## 2. Importazione librerie e configurazione

In [1]:
#IMPORTAZIONI

import os, shutil, time, random, string, logging, hashlib, tempfile, sys
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from datetime import datetime

#CONFIGURAZIONE LOGGING
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('SyncEase')


# WORKER MODULE CODE — funzioni pickle-able per ProcessPoolExecutor
_WORKER_MODULE_CODE = '''
import shutil
from pathlib import Path
from typing import Tuple

def copy_file_task(args: Tuple[str, str]) -> Tuple[bool, str]:
    """Copia un file da src a dst. Riceve stringhe per compatibilità pickle"""
    src_str, dst_str = args
    src = Path(src_str)
    dst = Path(dst_str)
    try:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        return True, str(dst)
    except Exception as e:
        return False, f"Errore copia {src} -> {dst}: {e}"

def delete_file_task(path_str: str) -> Tuple[bool, str]:
    """Elimina un file o directory. Riceve stringa per compatibilità pickle"""
    path = Path(path_str)
    try:
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()
        return True, str(path)
    except Exception as e:
        return False, f"Errore eliminazione {path}: {e}"
'''

_worker_path = Path(tempfile.gettempdir()) / 'syncease_workers.py'
_worker_path.write_text(_WORKER_MODULE_CODE)
if str(_worker_path.parent) not in sys.path:
    sys.path.insert(0, str(_worker_path.parent))

import syncease_workers
from syncease_workers import copy_file_task, delete_file_task

print("Librerie importate")
print("Worker module scritto su disco:", _worker_path)
print("Python multiprocessing ok")
print("Python multithreading ok")

Librerie importate
Worker module scritto su disco: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_workers.py
Python multiprocessing ok
Python multithreading ok


---
## 3. Modulo core: strutture dati e funzioni di supporto 
In questa sezione vengono definite le strutture dati fondamentali e le funzioni atomiche usate dal motore.

In [17]:
#STRUTTURE DATI

@dataclass
class FileInfo:
    """
    Rappresenta i metadati di un singolo file nel filesystem.
    Usata per confrontare i file sorgente e destinazione senza doverli leggere interamente — solo i metadati (stat) vengono
    esaminati nella fase di scansione, riducendo l'I/O al minimo
    """
    path: Path #path assoluto del file
    rel_path: str #path relativo alla cartella radice (chiave di confronto)
    size: int  #dimensione in byte
    mtime: float  #timestamp ultima modifica (Unix epoch)
    is_dir: bool = False  #true se è una directory


@dataclass
class SyncReport:
    """
    Report completo di una operazione di sincronizzazione.
    Raccoglie statistiche aggregate per consentire audit, debugging e valutazione delle performance.
    """
    source: str
    destination: str
    start_time: float = field(default_factory=time.time)
    end_time: float = 0.0

    # Contatori operazioni
    files_copied: int = 0 #file nuovi copiati
    files_updated: int = 0 #file aggiornati (sorgente più recente)
    files_deleted: int = 0  #file obsoleti rimossi
    dirs_created: int = 0  #directory create nella destinazione
    errors: List[str] = field(default_factory=list)  #errori non fatali

    @property
    def elapsed(self) -> float:
        """Tempo totale di esecuzione in secondi"""
        return self.end_time - self.start_time

    @property
    def total_operations(self) -> int:
        """Numero totale di operazioni effettuate"""
        return self.files_copied + self.files_updated + self.files_deleted

    def print_summary(self):
        """Stampa un riepilogo leggibile dell'operazione"""
        print(f"REPORT SINCRONIZZAZIONE")
        print(f"--Sorgente: {self.source}")
        print(f"--Destinazione: {self.destination}")
        print(f"--Durata: {self.elapsed:.3f}s")
        print(f">>File copiati: {self.files_copied}")
        print(f">>File aggiornati: {self.files_updated}")
        print(f">>File eliminati: {self.files_deleted}")
        print(f">>Dir create: {self.dirs_created}")
        print(f">>Errori: {len(self.errors)}")
        if self.errors:
            for err in self.errors:
                print(f">>> {err}")

print("Strutture dati definite: FileInfo, SyncReport")

Strutture dati definite: FileInfo, SyncReport


In [18]:
# FUNZIONI DI SUPPORTO

def scan_directory(root: Path) -> Dict[str, FileInfo]:
    """
    Scansiona ricorsivamente una directory e restituisce un dizionario path_relativo > FileInfo per ogni file e sottocartella trovata.
    Usa os.walk() che è ottimizzato dal sistema operativo per la navigazione ricorsiva. Il path relativo funge da chiave universale
    per confrontare sorgente e destinazione indipendentemente dal percorso assoluto

    Args:
        root: cartella radice da scansionare
    Returns:
        Dizionario {rel_path: FileInfo}
    """
    result: Dict[str, FileInfo] = {}

    for dirpath, dirnames, filenames in os.walk(root):
        current_dir = Path(dirpath)

        #registra le directory (utile per ricrearle nella destinazione)
        for dirname in dirnames:
            full_path = current_dir / dirname
            rel = str(full_path.relative_to(root))
            stat = full_path.stat()
            result[rel] = FileInfo(
                path=full_path,
                rel_path=rel,
                size=0,
                mtime=stat.st_mtime,
                is_dir=True
            )

        #registra i file con i loro metadati
        for filename in filenames:
            full_path = current_dir / filename
            rel = str(full_path.relative_to(root))
            try:
                stat = full_path.stat()
                result[rel] = FileInfo(
                    path=full_path,
                    rel_path=rel,
                    size=stat.st_size,
                    mtime=stat.st_mtime,
                    is_dir=False
                )
            except FileNotFoundError:
                #file rimosso durante la scansione (race condition)
                logger.warning(f"File sparito durante la scansione: {full_path}")

    return result


def scan_file_metadata(file_info: FileInfo) -> FileInfo:
    """
    Rilegge i metadati aggiornati di un file.
    Questa funzione è progettata per essere eseguita nei thread del pool sfruttando la concorrenza per velocizzare la lettura di molti file

    Args:
        file_info: FileInfo con path da aggiornare
    Returns:
        FileInfo aggiornato con i metadati correnti
    """
    try:
        stat = file_info.path.stat()
        file_info.size = stat.st_size
        file_info.mtime = stat.st_mtime
    except FileNotFoundError:
        pass  #il chiamante gestirà i FileInfo non aggiornati
    return file_info


def copy_file_task(args: Tuple[str, str]) -> Tuple[bool, str]:
    """
    Copia un singolo file. Riceve stringhe per compatibilità con pickle nel ProcessPoolExecutor su tutti i sistemi.
    """
    src_str, dst_str = args
    src = Path(src_str)
    dst = Path(dst_str)
    try:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        return True, str(dst)
    except Exception as e:
        return False, f"Errore copia {src} → {dst}: {e}"


def delete_file_task(path: Path) -> Tuple[bool, str]:
    """
    Elimina un file o una directory vuota dalla cartella di destinazione.

    Args:
        path: Path assoluto del file/directory da rimuovere
    Returns:
        (success: bool, message: str)
    """
    try:
        if path.is_dir():
            shutil.rmtree(path)  #rimuove ricorsivamente la directory
        else:
            path.unlink() # rimuove il file
        return True, str(path)
    except Exception as e:
        return False, f"Errore eliminazione {path}: {e}"

print("Funzioni helper definite: scan_directory, copy_file_task, delete_file_task")

Funzioni helper definite: scan_directory, copy_file_task, delete_file_task


---
## 4. Motore di sincronizzazione (SyncEngine)
In questa sezione viene presentato il cuore del sistema, che orchestra le tre fasi (scansione, categorizzazione, esecuzione).

In [19]:
#MOTORE DI SINCRONIZZAZIONE

def filter_redundant_deletions(to_delete: List[Path]) -> List[Path]:
    """
    Rimuove dalla lista i path che sono figli di una directory già presente nella stessa lista. Se scripts/ci viene eliminata con rmtree, 
    non serve eliminare anche scripts/ci/run_tests.sh dato che verrebbe già rimosso.
    Evita errori di race condition nel ThreadPoolExecutor
    """
    path_set = set(to_delete)
    result = []
    for p in to_delete:
        #se un antenato del path è già nella lista, questo path è ridondante
        redundant = any(parent in path_set and parent != p for parent in p.parents)
        if not redundant:
            result.append(p)
    return result

class SyncEngine:
    """
    Motore principale di sincronizzazione per SyncEase.
    Implementa una pipeline a tre fasi:
      1. SCANSIONE — legge i metadati di tutti i file con multithreading
      2. DIFF — confronta sorgente e destinazione, categorizza le operazioni
      3. ESECUZIONE — copia/elimina con multiprocessing, in chunk paralleli

    Parametri:
        max_workers_threads: numero di thread per la scansione I/O
        max_workers_procs: numero di processi per la copia parallela
        chunk_size: quanti file per chunk nel ProcessPoolExecutor
        tolerance_secs: tolleranza in secondi per il confronto mtime (utile per filesystem con diversa granularità)
    """

    def __init__(
        self,
        max_workers_threads: int = 8,
        max_workers_procs: int = None, #usa os.cpu_count()
        chunk_size: int = 50,
        tolerance_secs: float = 1.0
    ):
        self.max_workers_threads = max_workers_threads
        #se non specificato, usa tutti i core disponibili
        self.max_workers_procs = max_workers_procs or os.cpu_count() or 2
        self.chunk_size = chunk_size
        self.tolerance_secs = tolerance_secs

        logger.info(
            f"SyncEngine inizializzato: "
            f"{self.max_workers_threads} thread, "
            f"{self.max_workers_procs} processi, "
            f"chunk={self.chunk_size}"
        )

    #----------------------------------------------------------------------------------------

    #SCANSIONE

    def _scan_with_threads(self, root: Path) -> Dict[str, FileInfo]:
        """
        Scansiona la directory usando un pool di thread.

        Strategia:
          - scan_directory() ottiene la lista completa dei file (os.walk)
          - Il ThreadPoolExecutor rilegge in parallelo i metadati di ogni file aggiornando size e mtime in modo concorrente

        Il multithreading è efficace perché le operazioni stat() sono I/O-bound: il thread cede il controllo mentre aspetta il disco,
        permettendo ad altri thread di procedere contemporaneamente
        """
        logger.info(f"Scansione in corso: {root}")
        file_map = scan_directory(root)

        #separazione dei soli file (non directory) per l'aggiornamento parallelo
        files_only = [fi for fi in file_map.values() if not fi.is_dir]

        if files_only:
            with ThreadPoolExecutor(max_workers=self.max_workers_threads) as executor:
                #submit() crea un future per ogni file
                futures = {executor.submit(scan_file_metadata, fi): fi for fi in files_only}
                for future in as_completed(futures):
                    updated = future.result()
                    file_map[updated.rel_path] = updated

        logger.info(f"{len(file_map)} elementi trovati in {root.name}")
        return file_map

    #-----------------------------------------------------------------------

    #DIFF (categorizzazione)

    def _compute_diff(
        self,
        src_map: Dict[str, FileInfo],
        dst_map: Dict[str, FileInfo],
        src_root: Path,
        dst_root: Path
    ) -> Tuple[List, List, List, List]:
        """
        Confronta i due dizionari e restituisce quattro liste:
          - to_copy: (src_path, dst_path) — file nuovi da copiare
          - to_update: (src_path, dst_path) — file modificati da aggiornare
          - to_delete: dst_path — file obsoleti da eliminare
          - dirs_to_create: dst_path — directory mancanti nella dst

        Il confronto usa il timestamp mtime con una tolleranza per gestire filesystem con diversa granularità (per esempio fat32 ha precisione 2s)
        """
        to_copy = []
        to_update = []
        to_delete = []
        dirs_to_create = []

        #nuovi/modificati: scorrere la sorgente
        for rel, src_fi in src_map.items():
            dst_path = dst_root / rel

            if src_fi.is_dir:
                #directory: crearla nella destinazione se mancante
                if not dst_path.exists():
                    dirs_to_create.append(dst_path)
            else:
                if rel not in dst_map:
                    #il file non esiste nella destinazione - copia
                    to_copy.append((src_fi.path, dst_path))
                else:
                    dst_fi = dst_map[rel]
                    #la sorgente è più recente (con tolleranza) - aggiorna
                    if src_fi.mtime - dst_fi.mtime > self.tolerance_secs:
                        to_update.append((src_fi.path, dst_path))

        #obsoleti: file presenti nella dst ma assenti nella src
        for rel, dst_fi in dst_map.items():
            if rel not in src_map:
                to_delete.append(dst_root / rel)

        #ordina i to_delete per profondità decrescente (bisogna eliminare prima i file profondi, poi le cartelle padre)
        to_delete.sort(key=lambda p: len(p.parts), reverse=True)

        logger.info(
            f"Diff: {len(to_copy)} nuovi, {len(to_update)} aggiornamenti, "
            f"{len(to_delete)} eliminazioni, {len(dirs_to_create)} dir"
        )
        return to_copy, to_update, to_delete, dirs_to_create

#-----------------------------------------------------------------------

    #ESECUZIONE con Multiprocessing

    def _execute_copies(self, tasks: List[Tuple[Path, Path]], report: SyncReport, mode: str):
        if not tasks:
            return

        # Converti Path → str per compatibilità pickle con ProcessPoolExecutor
        str_tasks = [(str(src), str(dst)) for src, dst in tasks]

        logger.info(f"Avvio {mode} di {len(str_tasks)} file con {self.max_workers_procs} processi...")

        with ProcessPoolExecutor(max_workers=self.max_workers_procs) as executor:
            results = executor.map(
                syncease_workers.copy_file_task, #i miei run avvengono su macOS...
                str_tasks,
                chunksize=max(1, len(str_tasks) // (self.max_workers_procs * 4))
            )
            for success, message in results:
                if success:
                    if mode == 'copy':
                        report.files_copied += 1
                    else:
                        report.files_updated += 1
                else:
                    report.errors.append(message)
                    logger.error(message)

    def _execute_deletions(self, paths: List[Path], report: SyncReport):
        """
        Elimina i file obsoleti dalla cartella di destinazione.

        Usa ThreadPoolExecutor (non processi) perché le eliminazioni sono operazioni rapide e il costo di avviare nuovi processi supererebbe
        il beneficio. I thread condividono lo stesso spazio di memoria e hanno overhead di avvio quasi nullo.
        """
        if not paths:
            return
        logger.info(f"Eliminazione di {len(paths)} elementi obsoleti...")
        paths = filter_redundant_deletions(paths) 
        with ThreadPoolExecutor(max_workers=self.max_workers_threads) as executor:
            futures = {executor.submit(delete_file_task, p): p for p in paths}
            for future in as_completed(futures):
                success, message = future.result()
                if success:
                    report.files_deleted += 1
                else:
                    report.errors.append(message)
                    logger.error(message)

#-----------------------------------------------------------------------
    # METODO synchronize()

    def synchronize(self, source: str, destination: str) -> SyncReport:
        """
        Punto di ingresso principale: sincronizza source con destination.

        Pipeline completa:
          1. Valida i percorsi di input
          2. Scansiona entrambe le directory (multithreading)
          3. Calcola il diff tra le due mappe
          4. Crea directory mancanti
          5. Copia file nuovi (multiprocessing)
          6. Aggiorna file modificati (multiprocessing)
          7. Elimina file obsoleti (multithreading)
          8. Restituisce un SyncReport completo

        Args:
            source: percorso della cartella sorgente
            destination: percorso della cartella di destinazione
        Returns:
            SyncReport con statistiche complete
        """
        src_root = Path(source)
        dst_root = Path(destination)

        #validazione input
        if not src_root.exists():
            raise FileNotFoundError(f"Cartella sorgente non trovata: {source}")
        if not src_root.is_dir():
            raise NotADirectoryError(f"Il percorso sorgente non è una cartella: {source}")

        #crea la destinazione se non esiste
        dst_root.mkdir(parents=True, exist_ok=True)

        report = SyncReport(source=source, destination=destination)
        logger.info(f"Avvio sincronizzazione")
        logger.info(f">>SRC: {source}")
        logger.info(f">>DST: {destination}")

        #scansione parallela 
        logger.info("[1/4] Scansione directories...")
        src_map = self._scan_with_threads(src_root)
        dst_map = self._scan_with_threads(dst_root)

        #calcolo diff 
        logger.info("[2/4] Calcolo differenze...")
        to_copy, to_update, to_delete, dirs_to_create = self._compute_diff(
            src_map, dst_map, src_root, dst_root
        )

        #creazione directory (deve precedere la copia dei file) 
        logger.info("[3/4] Creazione struttura directory...")
        for d in sorted(dirs_to_create, key=lambda p: len(p.parts)):
            d.mkdir(parents=True, exist_ok=True)
            report.dirs_created += 1
            logger.debug(f">>Dir creata: {d}")

        #copia e aggiornamento con multiprocessing
        logger.info("[4/4] Esecuzione operazioni...")
        self._execute_copies(to_copy, report, mode='copy')
        self._execute_copies(to_update, report, mode='update')

        #eliminazione obsoleti
        self._execute_deletions(to_delete, report)

        report.end_time = time.time()
        logger.info(f"Sincronizzazione completata in {report.elapsed:.3f}s")

        return report


print("SyncEngine definito...ok")

SyncEngine definito...ok


---
## Funzioni di Supporto per i Test

Creiamo funzioni per generare scenari di test realistici.

In [20]:
#UTILITY PER I TEST

def create_test_file(path: Path, size_bytes: int = 1024, content: str = None):
    """
    Crea un file di test con contenuto casuale o specificato.

    Args:
        path: dove creare il file
        size_bytes: dimensione in byte (se content=None)
        content: contenuto esplicito (sovrascrive size_bytes)
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    if content is not None:
        path.write_text(content, encoding='utf-8')
    else:
        #genera dati pseudo-casuali per simulare file reali
        data = ''.join(random.choices(string.ascii_letters + string.digits + ' \n', k=size_bytes))
        path.write_text(data, encoding='utf-8')


def count_files(folder: Path) -> Tuple[int, int]:
    """
    Conta file e directory in una cartella (ricorsivo). Returns: (n_files, n_dirs)
    """
    n_files = n_dirs = 0
    for _, dirs, files in os.walk(folder):
        n_files += len(files)
        n_dirs += len(dirs)
    return n_files, n_dirs


def verify_sync(source: Path, destination: Path) -> Dict:
    """
    Verifica che la sincronizzazione sia avvenuta correttamente. Confronta il contenuto di sorgente e destinazione.

    Returns: Dizionario con risultati della verifica
    """
    src_map = scan_directory(source)
    dst_map = scan_directory(destination)

    # Considera solo i file (non directory) per il confronto
    src_files = {k: v for k, v in src_map.items() if not v.is_dir}
    dst_files = {k: v for k, v in dst_map.items() if not v.is_dir}

    missing_in_dst = set(src_files.keys()) - set(dst_files.keys())
    extra_in_dst = set(dst_files.keys()) - set(src_files.keys())

    #verifica contenuto per i file comuni
    content_mismatch = []
    for rel in set(src_files.keys()) & set(dst_files.keys()):
        src_content = (source / rel).read_bytes()
        dst_content = (destination / rel).read_bytes()
        if src_content != dst_content:
            content_mismatch.append(rel)

    is_perfect = (
        len(missing_in_dst) == 0 and
        len(extra_in_dst) == 0 and
        len(content_mismatch) == 0
    )

    return {
        'ok': is_perfect,
        'src_files': len(src_files),
        'dst_files': len(dst_files),
        'missing_in_dst': list(missing_in_dst),
        'extra_in_dst': list(extra_in_dst),
        'content_mismatch': content_mismatch
    }


def print_folder_tree(folder: Path, prefix: str = "", max_depth: int = 3, _depth: int = 0):
    """
    Stampa la struttura ad albero di una cartella. Utile per visualizzare il risultato della sincronizzazione
    """
    if _depth > max_depth:
        return
    try:
        entries = sorted(folder.iterdir(), key=lambda e: (e.is_file(), e.name))
    except PermissionError:
        return

    for i, entry in enumerate(entries):
        connector = "|__ " if i == len(entries) - 1 else "|-- "
        icon = "/cartella/" if entry.is_dir() else "/file/"
        size = f" ({entry.stat().st_size}B)" if entry.is_file() else ""
        print(f"{prefix}{connector}{icon} {entry.name}{size}")
        if entry.is_dir():
            extension = "    " if i == len(entries) - 1 else "│   "
            print_folder_tree(entry, prefix + extension, max_depth, _depth + 1)


print("Funzioni di test ok")

Funzioni di test ok


---
## 5. Caso di test 1 — cartella con pochi file
Scenario: Una piccola cartella di progetto con 5 file di testo.  
Obiettivo: Verificare copia iniziale, rilevamento modifiche e rimozione obsoleti.

In [21]:
#TEST 1 — SETUP: creazione cartella sorgente con pochi file

print("TEST 1: Cartella con pochi file")

#vengono usate directory temporanee per non inquinare il filesystem reale
test1_base = Path(tempfile.mkdtemp(prefix="syncease_test1_"))
src1 = test1_base / "source"
dst1 = test1_base / "destination"
src1.mkdir()
dst1.mkdir()

#creazione file di test con contenuto realistico
test_files_t1 = {
    "readme.txt": "# Progetto SyncEase\nDocumentazione principale.",
    "config.json": '{"version": "1.0", "debug": false, "max_workers": 8}',
    "data.csv": "id,nome,valore\n1,alpha,100\n2,beta,200\n3,gamma,300",
    "script.py": "# Script di elaborazione\ndef process(data):\n return data",
    "report.md": "## Report\n- Voce 1\n- Voce 2\n- Voce 3"
}

for filename, content in test_files_t1.items():
    create_test_file(src1 / filename, content=content)

print(f"\nCartella sorgente creata: {src1}")
print(f"Struttura iniziale:")
print_folder_tree(src1)

n_files, _ = count_files(src1)
print(f"\nTotale file nella sorgente: {n_files}")

TEST 1: Cartella con pochi file

Cartella sorgente creata: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test1_48n2f93j/source
Struttura iniziale:
|-- /file/ config.json (52B)
|-- /file/ data.csv (49B)
|-- /file/ readme.txt (46B)
|-- /file/ report.md (36B)
|__ /file/ script.py (56B)

Totale file nella sorgente: 5


In [22]:
# TEST 1a — PRIMA SINCRONIZZAZIONE (copia iniziale)

print("Fase 1a: Prima sincronizzazione (destinazione vuota)")

engine = SyncEngine(max_workers_threads=4, max_workers_procs=2)
report1a = engine.synchronize(str(src1), str(dst1))
report1a.print_summary()

#verifica della correttezza
result = verify_sync(src1, dst1)
if result['ok']:
    print(f"VERIFICA SUPERATA: tutti i {result['src_files']} file sono stati copiati correttamente.")
else:
    print(f"VERIFICA FALLITA: {result}")

print(f"\nStruttura destinazione dopo la sincronizzazione:")
print_folder_tree(dst1)

17:27:40 [INFO] SyncEngine inizializzato: 4 thread, 2 processi, chunk=50
17:27:40 [INFO] Avvio sincronizzazione
17:27:40 [INFO] >>SRC: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test1_48n2f93j/source
17:27:40 [INFO] >>DST: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test1_48n2f93j/destination
17:27:40 [INFO] [1/4] Scansione directories...
17:27:40 [INFO] Scansione in corso: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test1_48n2f93j/source
17:27:40 [INFO] 5 elementi trovati in source
17:27:40 [INFO] Scansione in corso: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test1_48n2f93j/destination
17:27:40 [INFO] 0 elementi trovati in destination
17:27:40 [INFO] [2/4] Calcolo differenze...
17:27:40 [INFO] Diff: 5 nuovi, 0 aggiornamenti, 0 eliminazioni, 0 dir
17:27:40 [INFO] [3/4] Creazione struttura directory...
17:27:40 [INFO] [4/4] Esecuzione operazioni...
17:27:40 [INFO] Avvio copy di 5 file con 2 processi...


Fase 1a: Prima sincronizzazione (destinazione vuota)


17:27:40 [INFO] Sincronizzazione completata in 0.121s


REPORT SINCRONIZZAZIONE
--Sorgente: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test1_48n2f93j/source
--Destinazione: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test1_48n2f93j/destination
--Durata: 0.121s
>>File copiati: 5
>>File aggiornati: 0
>>File eliminati: 0
>>Dir create: 0
>>Errori: 0
VERIFICA SUPERATA: tutti i 5 file sono stati copiati correttamente.

Struttura destinazione dopo la sincronizzazione:
|-- /file/ config.json (52B)
|-- /file/ data.csv (49B)
|-- /file/ readme.txt (46B)
|-- /file/ report.md (36B)
|__ /file/ script.py (56B)


In [23]:
# TEST 1b — SECONDA SINCRONIZZAZIONE (modifica + eliminazione)

print("Fase 1b: Simulazione modifiche nella sorgente")

#modifica un file esistente (aggiorniamo il contenuto e forziamo mtime futuro)
readme_path = src1 / "readme.txt"
readme_path.write_text("# Progetto SyncEase v2.0\nDocumentazione aggiornata con nuove funzionalità.")
# Impostiamo l'mtime nel futuro per garantire il rilevamento indipendentemente
# dalla granularità del filesystem del sistema host
future_time = time.time() + 10
os.utime(readme_path, (future_time, future_time))
print(f">>Modificato: readme.txt (mtime aggiornato a +10s)")

#aggiunge un file nuovo
create_test_file(src1 / "notes.txt", content="Appunti del meeting del 31/05/2026.")
print(f">>Aggiunto: notes.txt")

#elimina un file dalla sorgente (dovrà sparire anche dalla destinazione)
(src1 / "report.md").unlink()
print(f">>Eliminato dalla sorgente: report.md")

print(f"\nSorgente dopo le modifiche:")
print_folder_tree(src1)

print("\nEsecuzione seconda sincronizzazione...")
report1b = engine.synchronize(str(src1), str(dst1))
report1b.print_summary()

result = verify_sync(src1, dst1)
if result['ok']:
    print(f"VERIFICA SUPERATA: sorgente e destinazione sono identiche.")
else:
    print(f"VERIFICA FALLITA: {result}")

print(f"\nDestinazione dopo la seconda sincronizzazione:")
print_folder_tree(dst1)

17:27:40 [INFO] Avvio sincronizzazione
17:27:40 [INFO] >>SRC: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test1_48n2f93j/source
17:27:40 [INFO] >>DST: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test1_48n2f93j/destination
17:27:40 [INFO] [1/4] Scansione directories...
17:27:40 [INFO] Scansione in corso: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test1_48n2f93j/source
17:27:40 [INFO] 5 elementi trovati in source
17:27:40 [INFO] Scansione in corso: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test1_48n2f93j/destination
17:27:40 [INFO] 5 elementi trovati in destination
17:27:40 [INFO] [2/4] Calcolo differenze...
17:27:40 [INFO] Diff: 1 nuovi, 1 aggiornamenti, 1 eliminazioni, 0 dir
17:27:40 [INFO] [3/4] Creazione struttura directory...
17:27:40 [INFO] [4/4] Esecuzione operazioni...
17:27:40 [INFO] Avvio copy di 1 file con 2 processi...


Fase 1b: Simulazione modifiche nella sorgente
>>Modificato: readme.txt (mtime aggiornato a +10s)
>>Aggiunto: notes.txt
>>Eliminato dalla sorgente: report.md

Sorgente dopo le modifiche:
|-- /file/ config.json (52B)
|-- /file/ data.csv (49B)
|-- /file/ notes.txt (35B)
|-- /file/ readme.txt (75B)
|__ /file/ script.py (56B)

Esecuzione seconda sincronizzazione...


17:27:40 [INFO] Avvio update di 1 file con 2 processi...
17:27:40 [INFO] Eliminazione di 1 elementi obsoleti...
17:27:40 [INFO] Sincronizzazione completata in 0.175s


REPORT SINCRONIZZAZIONE
--Sorgente: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test1_48n2f93j/source
--Destinazione: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test1_48n2f93j/destination
--Durata: 0.175s
>>File copiati: 1
>>File aggiornati: 1
>>File eliminati: 1
>>Dir create: 0
>>Errori: 0
VERIFICA SUPERATA: sorgente e destinazione sono identiche.

Destinazione dopo la seconda sincronizzazione:
|-- /file/ config.json (52B)
|-- /file/ data.csv (49B)
|-- /file/ notes.txt (35B)
|-- /file/ readme.txt (75B)
|__ /file/ script.py (56B)


---
## 6. Caso di test 2 — cartella con molti file
Scenario: 500 file di dimensioni variabili per simulare un carico reale.  
Obiettivo: Verificare le performance del multiprocessing su grandi volumi.

In [24]:
# TEST 2 — SETUP: generazione massiva di file

print("TEST 2: Cartella con molti file (500 file)")

N_FILES = 500  #numero di file da generare

test2_base = Path(tempfile.mkdtemp(prefix="syncease_test2_"))
src2 = test2_base / "source"
dst2 = test2_base / "destination"
src2.mkdir()
dst2.mkdir()

#genera file con dimensioni variabili (512B - 8KB) per simulare un archivio reale
print(f"Generazione di {N_FILES} file casuali...")
file_sizes = []
for i in range(N_FILES):
    size = random.randint(512, 8192)  #da 512B a 8KB
    ext = random.choice(['.txt', '.log', '.csv', '.json', '.xml'])
    filename = f"file_{i:04d}{ext}"
    create_test_file(src2 / filename, size_bytes=size)
    file_sizes.append(size)

total_size_kb = sum(file_sizes) / 1024
print(f"\nStatistiche della sorgente:")
print(f">>File generati: {N_FILES}")
print(f">>Dimensione tot: {total_size_kb:.1f} KB")
print(f">>Media per file: {total_size_kb/N_FILES:.1f} KB")
print(f">>Min / Max: {min(file_sizes)/1024:.1f} KB / {max(file_sizes)/1024:.1f} KB")

TEST 2: Cartella con molti file (500 file)
Generazione di 500 file casuali...

Statistiche della sorgente:
>>File generati: 500
>>Dimensione tot: 2152.5 KB
>>Media per file: 4.3 KB
>>Min / Max: 0.5 KB / 8.0 KB


In [25]:
# TEST 2a — Benchmark: confronto singolo thread vs multiprocessing

#sincronizzazione con configurazione ottimizzata
print("\n Sincronizzazione con multiprocessing ottimizzato...")
engine_multi = SyncEngine(max_workers_threads=8, max_workers_procs=4, chunk_size=50)

t_start = time.perf_counter()
report2 = engine_multi.synchronize(str(src2), str(dst2))
t_multi = time.perf_counter() - t_start

report2.print_summary()

result2 = verify_sync(src2, dst2)
if result2['ok']:
    throughput = total_size_kb / max(report2.elapsed, 0.001)
    print(f"VERIFICA SUPERATA: {result2['src_files']} file sincronizzati correttamente.")
    print(f"Throughput stimato: {throughput:.1f} KB/s")
else:
    print(f"VERIFICA FALLITA: {result2}")

# TEST 2b — Idempotenza: seconda sincronizzazione senza modifiche
print("\n Verifica idempotenza (seconda sync senza modifiche)...")
report2b = engine_multi.synchronize(str(src2), str(dst2))
report2b.print_summary()

if report2b.total_operations == 0:
    print("IDEMPOTENZA VERIFICATA: nessuna operazione quando tutto è già sincronizzato.")
else:
    print(f"Attenzione: {report2b.total_operations} operazioni inattese nella seconda sync.")

17:27:40 [INFO] SyncEngine inizializzato: 8 thread, 4 processi, chunk=50
17:27:40 [INFO] Avvio sincronizzazione
17:27:40 [INFO] >>SRC: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test2_dunli5p5/source
17:27:40 [INFO] >>DST: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test2_dunli5p5/destination
17:27:40 [INFO] [1/4] Scansione directories...
17:27:40 [INFO] Scansione in corso: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test2_dunli5p5/source



 Sincronizzazione con multiprocessing ottimizzato...


17:27:40 [INFO] 500 elementi trovati in source
17:27:40 [INFO] Scansione in corso: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test2_dunli5p5/destination
17:27:40 [INFO] 0 elementi trovati in destination
17:27:40 [INFO] [2/4] Calcolo differenze...
17:27:40 [INFO] Diff: 500 nuovi, 0 aggiornamenti, 0 eliminazioni, 0 dir
17:27:40 [INFO] [3/4] Creazione struttura directory...
17:27:40 [INFO] [4/4] Esecuzione operazioni...
17:27:40 [INFO] Avvio copy di 500 file con 4 processi...
17:27:41 [INFO] Sincronizzazione completata in 0.211s


REPORT SINCRONIZZAZIONE
--Sorgente: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test2_dunli5p5/source
--Destinazione: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test2_dunli5p5/destination
--Durata: 0.211s
>>File copiati: 500
>>File aggiornati: 0
>>File eliminati: 0
>>Dir create: 0
>>Errori: 0


17:27:41 [INFO] Avvio sincronizzazione
17:27:41 [INFO] >>SRC: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test2_dunli5p5/source
17:27:41 [INFO] >>DST: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test2_dunli5p5/destination
17:27:41 [INFO] [1/4] Scansione directories...
17:27:41 [INFO] Scansione in corso: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test2_dunli5p5/source
17:27:41 [INFO] 500 elementi trovati in source
17:27:41 [INFO] Scansione in corso: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test2_dunli5p5/destination
17:27:41 [INFO] 500 elementi trovati in destination
17:27:41 [INFO] [2/4] Calcolo differenze...
17:27:41 [INFO] Diff: 0 nuovi, 0 aggiornamenti, 0 eliminazioni, 0 dir
17:27:41 [INFO] [3/4] Creazione struttura directory...
17:27:41 [INFO] [4/4] Esecuzione operazioni...
17:27:41 [INFO] Sincronizzazione completata in 0.046s


VERIFICA SUPERATA: 500 file sincronizzati correttamente.
Throughput stimato: 10204.1 KB/s

 Verifica idempotenza (seconda sync senza modifiche)...
REPORT SINCRONIZZAZIONE
--Sorgente: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test2_dunli5p5/source
--Destinazione: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test2_dunli5p5/destination
--Durata: 0.046s
>>File copiati: 0
>>File aggiornati: 0
>>File eliminati: 0
>>Dir create: 0
>>Errori: 0
IDEMPOTENZA VERIFICATA: nessuna operazione quando tutto è già sincronizzato.


In [26]:
# TEST 2c — Sincronizzazione incrementale: modifica 10% dei file

print("\n Sincronizzazione incrementale (modifica 10% dei file)...")

all_files = list(src2.glob("*.*"))
files_to_modify = random.sample(all_files, k=int(N_FILES * 0.1))  #10%
files_to_delete = random.sample(
    [f for f in all_files if f not in files_to_modify],
    k=int(N_FILES * 0.05)  #5%
)

#modifica i file selezionati
for f in files_to_modify:
    new_content = f.read_text() + "\n[AGGIORNATO]"
    f.write_text(new_content)
    future_mtime = time.time() + 10
    os.utime(f, (future_mtime, future_mtime))

#elimina i file selezionati
for f in files_to_delete:
    f.unlink()

#aggiunge nuovi file
n_new = int(N_FILES * 0.05)  # 5% nuovi
for i in range(n_new):
    create_test_file(src2 / f"new_file_{i:03d}.txt", size_bytes=random.randint(256, 2048))

print(f">> Modificati: {len(files_to_modify)} file")
print(f">>Eliminati: {len(files_to_delete)} file")
print(f">>Aggiunti: {n_new} file")

report2c = engine_multi.synchronize(str(src2), str(dst2))
report2c.print_summary()

result2c = verify_sync(src2, dst2)
status = "SUPERATA" if result2c['ok'] else "FALLITA"
print(f"Verifica: {status}")
print(f">>>Aggiornamenti attesi: {len(files_to_modify)} → Rilevati: {report2c.files_updated}")
print(f">>>Eliminazioni attese: {len(files_to_delete)} → Rilevate: {report2c.files_deleted}")
print(f">>>Nuovi attesi: {n_new} → Rilevati: {report2c.files_copied}")

17:27:41 [INFO] Avvio sincronizzazione
17:27:41 [INFO] >>SRC: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test2_dunli5p5/source
17:27:41 [INFO] >>DST: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test2_dunli5p5/destination
17:27:41 [INFO] [1/4] Scansione directories...
17:27:41 [INFO] Scansione in corso: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test2_dunli5p5/source
17:27:41 [INFO] 500 elementi trovati in source
17:27:41 [INFO] Scansione in corso: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test2_dunli5p5/destination
17:27:41 [INFO] 500 elementi trovati in destination
17:27:41 [INFO] [2/4] Calcolo differenze...
17:27:41 [INFO] Diff: 25 nuovi, 50 aggiornamenti, 25 eliminazioni, 0 dir
17:27:41 [INFO] [3/4] Creazione struttura directory...
17:27:41 [INFO] [4/4] Esecuzione operazioni...
17:27:41 [INFO] Avvio copy di 25 file con 4 processi...



 Sincronizzazione incrementale (modifica 10% dei file)...
>> Modificati: 50 file
>>Eliminati: 25 file
>>Aggiunti: 25 file


17:27:41 [INFO] Avvio update di 50 file con 4 processi...
17:27:41 [INFO] Eliminazione di 25 elementi obsoleti...
17:27:41 [INFO] Sincronizzazione completata in 0.280s


REPORT SINCRONIZZAZIONE
--Sorgente: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test2_dunli5p5/source
--Destinazione: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test2_dunli5p5/destination
--Durata: 0.280s
>>File copiati: 25
>>File aggiornati: 50
>>File eliminati: 25
>>Dir create: 0
>>Errori: 0
Verifica: SUPERATA
>>>Aggiornamenti attesi: 50 → Rilevati: 50
>>>Eliminazioni attese: 25 → Rilevate: 25
>>>Nuovi attesi: 25 → Rilevati: 25


---
## 7. Caso di Test 3 — Struttura con Sottocartelle <a id='test3'></a>

**Scenario:** Struttura gerarchica realistica che simula un repository aziendale con 4 livelli di profondità.  
**Obiettivo:** Verificare la corretta gestione ricorsiva e la preservazione della struttura delle directory.

In [27]:
# TEST 3 — SETUP: struttura gerarchica realistica

print("TEST 3: Struttura con sottocartelle profonde")

test3_base = Path(tempfile.mkdtemp(prefix="syncease_test3_"))
src3 = test3_base / "source"
dst3 = test3_base / "destination"
src3.mkdir()
dst3.mkdir()

#struttura che simula un progetto aziendale reale
structure = {
    #root
    "README.md": "# SyncEase Enterprise\nVersione 3.0",
    "LICENSE.txt": "MIT License\nCopyright 2026 SyncEase",
    ".gitignore": "__pycache__/\n*.pyc\n.env\n",

    #sorgente applicazione
    "src/__init__.py": "# Pacchetto principale",
    "src/main.py": "from engine import SyncEngine\n\nif __name__ == '__main__':\n    SyncEngine().run()",
    "src/engine.py": "class SyncEngine:\n    def run(self):\n        pass",
    "src/utils/helpers.py": "def format_size(b):\n    return f'{b/1024:.1f} KB'",
    "src/utils/logger.py": "import logging\nlogger = logging.getLogger('app')",
    "src/models/file_info.py": "from dataclasses import dataclass\n@dataclass\nclass FileInfo: pass",
    "src/models/report.py": "from dataclasses import dataclass\n@dataclass\nclass Report: pass",

    #test
    "tests/test_engine.py": "import pytest\ndef test_sync(): assert True",
    "tests/test_utils.py": "import pytest\ndef test_helpers(): assert True",
    "tests/fixtures/sample.txt": "Sample data for testing.",
    "tests/fixtures/data.json": '{"test": true, "items": [1, 2, 3]}',

    #documentazione
    "docs/architecture.md": "## Architettura\n### Multiprocessing\n...",
    "docs/api/reference.md": "## API Reference\n### SyncEngine\n...",
    "docs/api/examples.md": "## Esempi\n```python\nengine.sync(src, dst)\n```",
    "docs/guides/quickstart.md": "## Quick Start\n1. Installa\n2. Configura\n3. Esegui",

    #configurazioni e dati
    "config/production.json": '{"workers": 8, "log_level": "INFO"}',
    "config/development.json": '{"workers": 2, "log_level": "DEBUG"}',
    "data/raw/dataset_2026.csv": "timestamp,value,status\n2026-01-01,42,OK\n2026-01-02,37,OK",
    "data/processed/output.csv": "date,total\n2026-01,79",
    "data/archives/backup_q1.csv": "q1_data",
    "data/archives/backup_q2.csv": "q2_data",

    #script di deployment
    "scripts/deploy.sh": "#!/bin/bash\necho 'Deploying SyncEase...'",
    "scripts/build.sh": "#!/bin/bash\npython setup.py build",
    "scripts/ci/run_tests.sh": "#!/bin/bash\npytest tests/ -v",
}

for rel_path, content in structure.items():
    create_test_file(src3 / rel_path, content=content)

n_files, n_dirs = count_files(src3)
print(f"Struttura sorgente creata:")
print_folder_tree(src3, max_depth=4)
print(f"\nTotale: {n_files} file in {n_dirs} directory")

TEST 3: Struttura con sottocartelle profonde
Struttura sorgente creata:
|-- /cartella/ config
│   |-- /file/ development.json (36B)
│   |__ /file/ production.json (35B)
|-- /cartella/ data
│   |-- /cartella/ archives
│   │   |-- /file/ backup_q1.csv (7B)
│   │   |__ /file/ backup_q2.csv (7B)
│   |-- /cartella/ processed
│   │   |__ /file/ output.csv (21B)
│   |__ /cartella/ raw
│       |__ /file/ dataset_2026.csv (56B)
|-- /cartella/ docs
│   |-- /cartella/ api
│   │   |-- /file/ examples.md (45B)
│   │   |__ /file/ reference.md (35B)
│   |-- /cartella/ guides
│   │   |__ /file/ quickstart.md (49B)
│   |__ /file/ architecture.md (39B)
|-- /cartella/ scripts
│   |-- /cartella/ ci
│   │   |__ /file/ run_tests.sh (28B)
│   |-- /file/ build.sh (33B)
│   |__ /file/ deploy.sh (40B)
|-- /cartella/ src
│   |-- /cartella/ models
│   │   |-- /file/ file_info.py (65B)
│   │   |__ /file/ report.py (63B)
│   |-- /cartella/ utils
│   │   |-- /file/ helpers.py (49B)
│   │   |__ /file/ logger.py (48B)

In [28]:
# TEST 3a — Prima sincronizzazione: verifica struttura ricorsiva

print("\nPrima sincronizzazione (struttura gerarchica completa)...")
engine3 = SyncEngine(max_workers_threads=6, max_workers_procs=2)
report3a = engine3.synchronize(str(src3), str(dst3))
report3a.print_summary()

result3 = verify_sync(src3, dst3)
print(f"\nStruttura destinazione dopo la sincronizzazione:")
print_folder_tree(dst3, max_depth=4)

if result3['ok']:
    print(f"\nVERIFICA SUPERATA")
    print(f">>File sincronizzati correttamente: {result3['src_files']}")
    print(f">>Directory ricreate: {report3a.dirs_created}")
else:
    print(f"\nVERIFICA FALLITA:")
    if result3['missing_in_dst']:
        print(f">>Mancanti: {result3['missing_in_dst']}")
    if result3['content_mismatch']:
        print(f">>Contenuto diverso: {result3['content_mismatch']}")

17:27:41 [INFO] SyncEngine inizializzato: 6 thread, 2 processi, chunk=50
17:27:41 [INFO] Avvio sincronizzazione
17:27:41 [INFO] >>SRC: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test3_j5osu7yi/source
17:27:41 [INFO] >>DST: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test3_j5osu7yi/destination
17:27:41 [INFO] [1/4] Scansione directories...
17:27:41 [INFO] Scansione in corso: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test3_j5osu7yi/source
17:27:41 [INFO] 42 elementi trovati in source
17:27:41 [INFO] Scansione in corso: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test3_j5osu7yi/destination
17:27:41 [INFO] 0 elementi trovati in destination
17:27:41 [INFO] [2/4] Calcolo differenze...
17:27:41 [INFO] Diff: 27 nuovi, 0 aggiornamenti, 0 eliminazioni, 15 dir
17:27:41 [INFO] [3/4] Creazione struttura directory...
17:27:41 [INFO] [4/4] Esecuzione operazioni...
17:27:41 [INFO] Avvio copy di 27 file con 2 processi...



Prima sincronizzazione (struttura gerarchica completa)...


17:27:41 [INFO] Sincronizzazione completata in 0.091s


REPORT SINCRONIZZAZIONE
--Sorgente: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test3_j5osu7yi/source
--Destinazione: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test3_j5osu7yi/destination
--Durata: 0.091s
>>File copiati: 27
>>File aggiornati: 0
>>File eliminati: 0
>>Dir create: 15
>>Errori: 0

Struttura destinazione dopo la sincronizzazione:
|-- /cartella/ config
│   |-- /file/ development.json (36B)
│   |__ /file/ production.json (35B)
|-- /cartella/ data
│   |-- /cartella/ archives
│   │   |-- /file/ backup_q1.csv (7B)
│   │   |__ /file/ backup_q2.csv (7B)
│   |-- /cartella/ processed
│   │   |__ /file/ output.csv (21B)
│   |__ /cartella/ raw
│       |__ /file/ dataset_2026.csv (56B)
|-- /cartella/ docs
│   |-- /cartella/ api
│   │   |-- /file/ examples.md (45B)
│   │   |__ /file/ reference.md (35B)
│   |-- /cartella/ guides
│   │   |__ /file/ quickstart.md (49B)
│   |__ /file/ architecture.md (39B)
|-- /cartella/ scripts
│   |-- /cartella/ ci
│   │   |__

In [29]:
# TEST 3b — Ristrutturazione: riorganizzazione del progetto

print("\nSimulazione ristrutturazione del progetto...")

#spostamento: rinomina una cartella (eliminazione + creazione)
old_scripts_ci = src3 / "scripts" / "ci"
new_scripts_ci = src3 / "scripts" / "pipelines"
if old_scripts_ci.exists():
    shutil.copytree(old_scripts_ci, new_scripts_ci)
    shutil.rmtree(old_scripts_ci)
    print(f">>Rinominato: scripts/ci → scripts/pipelines")

#nuova funzionalità: aggiunta cartella 'monitoring'
monitoring_files = {
    "src/monitoring/__init__.py": "# Monitoring module",
    "src/monitoring/metrics.py": "class MetricsCollector:\n  def collect(self): pass",
    "src/monitoring/alerts.py": "class AlertManager:\n  def send(self, msg): pass",
}
for rel, content in monitoring_files.items():
    create_test_file(src3 / rel, content=content)
print(f">>>Aggiunto modulo: src/monitoring/ (3 file)")

#deprecazione: rimozione cartella data/archives
archives_dir = src3 / "data" / "archives"
if archives_dir.exists():
    shutil.rmtree(archives_dir)
    print(f">>>Rimossa: data/archives/ (deprecata)")

#aggiornamento file principale
main_path = src3 / "src" / "main.py"
main_path.write_text("from engine import SyncEngine\nfrom monitoring.metrics import MetricsCollector\n\nif __name__ == '__main__':\n    mc = MetricsCollector()\n    SyncEngine().run()")
future_mtime = time.time() + 5
os.utime(main_path, (future_mtime, future_mtime))
print(f">>>Aggiornato: src/main.py")

print("\nSeconda sincronizzazione (ristrutturazione)...")
report3b = engine3.synchronize(str(src3), str(dst3))
report3b.print_summary()

result3b = verify_sync(src3, dst3)
status = "SUPERATA" if result3b['ok'] else "FALLITA"
print(f"Verifica: {status}")

print(f"\nDestinazione dopo la ristrutturazione:")
print_folder_tree(dst3, max_depth=4)

17:27:41 [INFO] Avvio sincronizzazione
17:27:41 [INFO] >>SRC: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test3_j5osu7yi/source
17:27:41 [INFO] >>DST: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test3_j5osu7yi/destination
17:27:41 [INFO] [1/4] Scansione directories...
17:27:41 [INFO] Scansione in corso: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test3_j5osu7yi/source
17:27:41 [INFO] 43 elementi trovati in source
17:27:41 [INFO] Scansione in corso: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test3_j5osu7yi/destination
17:27:42 [INFO] 42 elementi trovati in destination
17:27:42 [INFO] [2/4] Calcolo differenze...
17:27:42 [INFO] Diff: 4 nuovi, 1 aggiornamenti, 5 eliminazioni, 2 dir
17:27:42 [INFO] [3/4] Creazione struttura directory...
17:27:42 [INFO] [4/4] Esecuzione operazioni...
17:27:42 [INFO] Avvio copy di 4 file con 2 processi...



Simulazione ristrutturazione del progetto...
>>Rinominato: scripts/ci → scripts/pipelines
>>>Aggiunto modulo: src/monitoring/ (3 file)
>>>Rimossa: data/archives/ (deprecata)
>>>Aggiornato: src/main.py

Seconda sincronizzazione (ristrutturazione)...


17:27:42 [INFO] Avvio update di 1 file con 2 processi...
17:27:42 [INFO] Eliminazione di 5 elementi obsoleti...
17:27:42 [INFO] Sincronizzazione completata in 0.158s


REPORT SINCRONIZZAZIONE
--Sorgente: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test3_j5osu7yi/source
--Destinazione: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test3_j5osu7yi/destination
--Durata: 0.158s
>>File copiati: 4
>>File aggiornati: 1
>>File eliminati: 2
>>Dir create: 2
>>Errori: 0
Verifica: SUPERATA

Destinazione dopo la ristrutturazione:
|-- /cartella/ config
│   |-- /file/ development.json (36B)
│   |__ /file/ production.json (35B)
|-- /cartella/ data
│   |-- /cartella/ processed
│   │   |__ /file/ output.csv (21B)
│   |__ /cartella/ raw
│       |__ /file/ dataset_2026.csv (56B)
|-- /cartella/ docs
│   |-- /cartella/ api
│   │   |-- /file/ examples.md (45B)
│   │   |__ /file/ reference.md (35B)
│   |-- /cartella/ guides
│   │   |__ /file/ quickstart.md (49B)
│   |__ /file/ architecture.md (39B)
|-- /cartella/ scripts
│   |-- /cartella/ pipelines
│   │   |__ /file/ run_tests.sh (28B)
│   |-- /file/ build.sh (33B)
│   |__ /file/ deploy.sh (40B)
|-

---
## 8. Report finale e riepilogo

In [30]:
# SEZIONE FINALE: RIEPILOGO COMPLETO DI TUTTI I TEST

print("RIEPILOGO COMPLETO — SYNCEASE TEST SUITE")

all_reports = [
    ("Test 1a", "Pochi file — Copia iniziale", report1a),
    ("Test 1b", "Pochi file — Modifica/Eliminazione", report1b),
    ("Test 2a", "500 file  — Prima sync", report2),
    ("Test 2b", "500 file  — Idempotenza", report2b),
    ("Test 2c", "500 file  — Sync incrementale", report2c),
    ("Test 3a", "Sottocart — Struttura iniziale", report3a),
    ("Test 3b", "Sottocart — Ristrutturazione", report3b),
]

print(f"\n{'ID':<8} {'Descrizione':<35} {'Copiati':>8} {'Agg.':>6} {'Elim.':>6} {'Tempo':>8} {'Errori':>7}")

total_time = 0
total_ops = 0
total_errors = 0

for test_id, desc, report in all_reports:
    total_time += report.elapsed
    total_ops += report.total_operations
    total_errors += len(report.errors)
    print(
        f"{test_id:<8} {desc:<35} "
        f"{report.files_copied:>8} "
        f"{report.files_updated:>6} "
        f"{report.files_deleted:>6} "
        f"{report.elapsed:>7.3f}s "
        f"{len(report.errors):>7}"
    )

print(f"{'TOTALE':<44} {total_ops:>8} operazioni in {total_time:.3f}s | {total_errors} errori")

print("\nConclusioni:")
print(f">>ok...Tutti i test completati senza errori critici")
print(f">>ok...Idempotenza verificata (sync già sincronizzata = 0 operazioni)")
print(f">>ok...Struttura ricorsiva gestita correttamente")
print(f">>ok...Rilevamento modifiche tramite mtime funzionante")
print(f">>ok...Eliminazione file obsoleti funzionante")
print(f">>ok...{N_FILES} file gestiti con multiprocessing in {report2.elapsed:.3f}s")

RIEPILOGO COMPLETO — SYNCEASE TEST SUITE

ID       Descrizione                          Copiati   Agg.  Elim.    Tempo  Errori
Test 1a  Pochi file — Copia iniziale                5      0      0   0.121s       0
Test 1b  Pochi file — Modifica/Eliminazione         1      1      1   0.175s       0
Test 2a  500 file  — Prima sync                   500      0      0   0.211s       0
Test 2b  500 file  — Idempotenza                    0      0      0   0.046s       0
Test 2c  500 file  — Sync incrementale             25     50     25   0.280s       0
Test 3a  Sottocart — Struttura iniziale            27      0      0   0.091s       0
Test 3b  Sottocart — Ristrutturazione               4      1      2   0.158s       0
TOTALE                                            642 operazioni in 1.082s | 0 errori

Conclusioni:
>>ok...Tutti i test completati senza errori critici
>>ok...Idempotenza verificata (sync già sincronizzata = 0 operazioni)
>>ok...Struttura ricorsiva gestita correttamente
>>ok...

In [16]:
# CLEANUP: rimozione delle directory temporanee di test
#rimozione delle directory temporanee per liberare spazio.

for test_dir in [test1_base, test2_base, test3_base]:
    try:
        shutil.rmtree(test_dir)
        print(f"Rimossa directory temporanea: {test_dir}")
    except Exception as e:
        print(f"Impossibile rimuovere {test_dir}: {e}")

Rimossa directory temporanea: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test1_9hevv97g
Rimossa directory temporanea: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test2_5odt_o5s
Rimossa directory temporanea: /var/folders/xd/gg9fwq69289958mc26w3vmbw0000gn/T/syncease_test3_ahco8zo5
